<img src="https://upload.wikimedia.org/wikipedia/commons/3/35/Uba_fiuba_ingenieria_logo.png" width="300" align="center">



# **Analisis de Series de Tiempo II**

# **Clase 3, LSTM, GRU y Benchmark contra el Naive**

Importamos las librerías necesarias

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import urllib.request
from scipy.io import wavfile
from torch.utils.data import DataLoader, TensorDataset

Fijamos las semillas para reproducibilidad

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

Descargamos y preparamos el audio

In [ ]:
URL = "https://raw.githubusercontent.com/pytorch/audio/main/test/torchaudio_unittest/assets/steam-train-whistle-daniel_simon.wav"
urllib.request.urlretrieve(URL, "tren.wav")
sr, w = wavfile.read("tren.wav")
audio = w.mean(axis=1).astype(np.float32)   # convertimos a mono
audio = audio[::4]  # submuestreamos a 11,025 Hz
sr = sr // 4
audio = audio / np.abs(audio).max()  # normalizamos a [-1, 1]

Construimos las ventanas de train y test con split temporal

In [ ]:
L, H = 160, 1
n_tr = int(len(audio) * 0.8)
X_tr, y_tr, X_te, y_te = [], [], [], []
for i in range(0, n_tr - L - H + 1, 4):
    X_tr.append(audio[i : i + L]); y_tr.append(audio[i + L : i + L + H])
for i in range(n_tr - L, len(audio) - L - H + 1, 4):
    X_te.append(audio[i : i + L]); y_te.append(audio[i + L : i + L + H])
X_tr = torch.tensor(np.array(X_tr)).unsqueeze(-1); y_tr = torch.tensor(np.array(y_tr))
X_te = torch.tensor(np.array(X_te)).unsqueeze(-1); y_te = torch.tensor(np.array(y_te))

Contamos los parámetros de cada celda: el precio de las compuertas

In [ ]:
for nombre, tipo in [("RNN", nn.RNN), ("GRU", nn.GRU), ("LSTM", nn.LSTM)]:
    celda = tipo(1, 64, batch_first=True)  # creamos la celda con d_h=64
    total = sum(p.numel() for p in celda.parameters())  # sumamos todos sus parámetros
    print(f"{nombre:5s} (d_h=64): {total:>6,d} parámetros")

RNN   (d_h=64):  4,288 parámetros
GRU   (d_h=64): 12,864 parámetros
LSTM  (d_h=64): 17,152 parámetros


Calculamos el baseline naive: la siguiente muestra = la muestra actual

In [ ]:
naive = ((X_te[:, -1, 0:1] - y_te)**2).mean().sqrt().item()
print(f"\n{'modelo':8s} RMSE next-sample")
print(f"{'naive':8s} {naive:.4f}")


modelo   RMSE next-sample
naive    0.0796


Entrenamos las tres celdas con el mismo ciclo: solo cambia la celda

In [ ]:
for nombre, tipo in [("RNN", nn.RNN), ("GRU", nn.GRU), ("LSTM", nn.LSTM)]:
    torch.manual_seed(42)   # misma semilla para comparar justo
    rnn = tipo(1, 64, batch_first=True)  # creamos la celda recurrente
    cabeza = nn.Linear(64, 1)  # definimos la cabeza de salida
    parametros = list(rnn.parameters()) + list(cabeza.parameters())
    optimizador = torch.optim.Adam(parametros, lr=3e-3)  # definimos el optimizador
    lotes = DataLoader(TensorDataset(X_tr, y_tr), batch_size=256, shuffle=True)
    for epoca in range(5):   # entrenamos 5 épocas
        for xb, yb in lotes:   # recorremos los minibatches
            optimizador.zero_grad()  # limpiamos los gradientes
            salida, _ = rnn(xb)   # la celda recorre las ventanas
            prediccion = cabeza(salida[:, -1])  # predecimos desde el último estado
            perdida = ((prediccion - yb)**2).mean()  # calculamos el MSE
            perdida.backward()    # propagamos el gradiente
            nn.utils.clip_grad_norm_(parametros, 1.0)    # aplicamos gradient clipping (siempre)
            optimizador.step()   # actualizamos los pesos
    with torch.no_grad():   # evaluamos sin gradientes
        salida, _ = rnn(X_te)
        rmse = ((cabeza(salida[:, -1]) - y_te)**2).mean().sqrt().item()
    print(f"{nombre:8s} {rmse:.4f}")

RNN      0.0323
GRU      0.0311
LSTM     0.0381


Conclusión: en los retornos mensuales el naive competía; aquí las recurrentes lo dejan atrás porque hay señal secuencial densa que aprender.